In [28]:
!pip install -U langchain_community --quiet

In [29]:
from langchain_community.document_loaders import TextLoader
%pip install "unstructured[xlsx]"
from langchain_community.document_loaders import UnstructuredExcelLoader
loader_text = TextLoader("Company_sample.txt")
loader_excel = UnstructuredExcelLoader("Company_sample.xlsx")
docs_text = loader_text.load()
docs_excel = loader_excel.load()
docs = docs_text + docs_excel


In [30]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 50)
chunks = splitter.split_documents(docs)

In [46]:
import os
os.environ["GOOGLE_API_KEY"] = "your_api_key_here"

In [44]:
%pip install -U langchain-google-genai --quiet
import os
from langchain_google_genai import GoogleGenerativeAIEmbeddings
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY
)

In [32]:
%pip install faiss-cpu --quiet
from langchain_community.vectorstores import FAISS
vector_store = FAISS.from_documents(chunks, embeddings)

In [33]:
retriever = vector_store.as_retriever(search_type="similarity", k = 3)

In [34]:
from langchain_core.prompts import PromptTemplate
prompt_template = PromptTemplate(

    input_variables = {"context", "question"},

    template = """You are a helpful assistant that answers questions based on the provided context.

    Context:
    {context}

    Question: {question}

    Answer: Provide a clear and concise answer based on the context above, if the context doesn't contain enough information to answer the answer then say so"""

)

In [45]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash",
    google_api_key = GOOGLE_API_KEY,
    temperature = 0.5,
    convert_system_message_to_human = True
)
response = llm.invoke("what are the core values of ACME Corporation as given in the txt file?")
print(response.content)

I'm sorry, but I cannot access local files on your computer, including any `txt` file you might be referring to. My capabilities are limited to the information I was trained on and what you provide directly in our conversation.

Therefore, I cannot tell you the specific core values of ACME Corporation as given in *your* `txt` file.

**To find the core values, you would need to open the `txt` file yourself and look for sections related to:**
*   Company mission
*   Vision statement
*   Values
*   Culture
*   Guiding principles


In [38]:
def rag(query):
  docs = retriever.invoke(query)
  context = "\n\n".join([doc.page_content for doc in docs])
  prompt = prompt_template.format(context=context, question=query)
  response = llm.invoke(prompt)
  answer = response.content
  return answer
rag("how many employees are there in the sales department?")

'There are 18 employees in the Sales department.'